In [ ]:
### Author: Demesvar D. Destin
### Year: 2026
### Contact: ddd5579@psu.edu/demesvardestin@gmail.com

In [ ]:
### BEGIN IMPORT NEEDED LIBRARIES ###
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats import chisquare
from sklearn.metrics import roc_curve, roc_auc_score
from datetime import datetime
import math
import os
### END IMPORT NEEDED LIBRARIES ###

### BEGIN SETTING DATASET VARIABLES AND CREATING PANDAS ###

print ("Downloading dataframes from AoU...")

## NOTE: In AoU Researcher Workbench, dataframes (dfs) are downloaded directly using SQL,
## but these are not included here due to data privacy rules. all dfs named below refer to
## the downloaded data that stayed in the Workbench.
## person_df: demographics
## survey_df: alcohol, smoking
## measurement_df: anthropometrics
## drug_df: medications
## condition_df: joint pain status

# drop duplicates from survey_df and condition_df
# keep duplicates from drug_df (ie. if taking AIT and painkiller, would be two rows)
# keep duplicates from measurement_df (ie. one person's bmi and height would be two rows)
survey_df.drop_duplicates(subset=['person_id'], inplace=True)
condition_df = condition_df.sort_values('condition_start_datetime', ascending=False)
condition_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

# add age, drinks_per_year, smoke_years, height,
# hip_circumference, waist_circumference, bri, bmi columns
# to person_df, survey_df, and measurement_df, respectively
person_df.insert(loc=4, column='age', value=0)
measurement_df.insert(loc=3, column='height', value=0)
measurement_df.insert(loc=4, column='hip_circumference', value=0)
measurement_df.insert(loc=5, column='waist_circumference', value=0)
measurement_df.insert(loc=6, column='bri', value=0)
measurement_df.insert(loc=7, column='bmi', value=0)

# rename 'standard_concept_name' column in condition_df to
# pain (aka joint pain) since we've only selected joint pain as a condition
condition_df.rename(columns={'standard_concept_name': 'pain'}, inplace=True)

# rename 'standard_concept_name' column in drug_df in drugs_df to drugs
# since that's the information that's contained
drug_df.rename(columns={'standard_concept_name': 'drugs'}, inplace=True)

drugs_to_keep = ['anastrozole', 'letrozole', 'exemestane', 'tamoxifen', 'paclitaxel', 'cabazitaxel', 'docetaxel']
pattern = '|'.join(drugs_to_keep)
drug_df = drug_df[drug_df['drugs'].str.contains(pattern, case=False, na=False)]

In [ ]:
ait_to_keep = '|'.join(['anastrozole', 'letrozole', 'exemestane'])
others = '|'.join(['tamoxifen', 'paclitaxel', 'cabazitaxel', 'docetaxel'])

ait_df = drug_df[drug_df['drugs'].str.contains(ait_to_keep, case=False, na=False)]
other_drugs_df = drug_df[drug_df['drugs'].str.contains(others, case=False, na=False)]

In [ ]:
ait_df = ait_df.sort_values('drug_exposure_start_datetime', ascending=False)
ait_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

other_drugs_df.drop_duplicates(subset=['person_id'], inplace=True, keep='first')

In [ ]:
import pandas as pd
import numpy as np
import math
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from scipy.stats import chisquare
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

### BEGIN INITIAL DATA CLEANUP AND TRANSFORMATION ###

print("Initiating data cleanup...")

# function to calculate age at treatment initiation
# takes date of birth (born) and AIT start date
def calculate_age(born, ait_start_date):
    try:
        birthday = born.replace(year = int(ait_start_date.year))
    # raised when birth date is February 29
    # and the current year is not a leap year
    except ValueError:
        birthday = born.replace(year = int(ait_start_date.year),
                  month = born.month + 1, day = 1)
    if birthday > ait_start_date:
        return ait_start_date.year - born.year - 1
    else:
        return ait_start_date.year - born.year

# function to calculate body roundness index (BRI)
# takes waist_circumference and height as arguments
def calculate_bri(waist_circumference_cm, height_cm):

    # validate args to ensure positive numbers
    if waist_circumference_cm <= 0 or height_cm <= 0:
        return 0
    try:
        # core ratio term
        first_term = (waist_circumference_cm / (2 * math.pi))
        second_term = (0.5 * height_cm)
        ratio_term = (first_term*first_term) / (second_term*second_term)

        # check if the value inside the square root is non-negative
        value_under_sqrt = 1 - ratio_term

        # apply BRI formula
        bri = 364.2 - 365.5 * value_under_sqrt
        return bri

    except ZeroDivisionError:
        print("error: height cannot be zero")
        return 0
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return 0

# create empty anthropometric df to collapse duplicate
# measurement_df rows into unique rows
anthro_data = {'person_id': '', 'height': 0.0, 'waist_circumference': 0.0,
               'hip_circumference': 0.0, 'cars': 0.0, 'bmi': 0.0, 'bri': 0.0, 'whr': 0.0}
anthro_df = pd.DataFrame(anthro_data, index=[0])

# loop through measurement_df, transfer height, weight,
# waist circumference, hip_circumference data to anthro_df
# based on unique person_id
for index, row in measurement_df.iterrows():
    if len(anthro_df.loc[anthro_df['person_id'] == row['person_id']]) == 0:
        new_row_df = pd.DataFrame([{'person_id': row['person_id'], 'height': 0.0,
                                    'waist_circumference': 0.0, 'hip_circumference': 0.0,
                                    'bmi': 0.0, 'bri': 0.0, 'whr': 0.0, 'cars': 0.0}])
        anthro_df = pd.concat([anthro_df, new_row_df], ignore_index=True)

    if pd.isna(row['value_as_number']):
        measurement_df.loc[index, 'value_as_number'] = 0.0

    if 'height' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'height'] = row['value_as_number']

    if 'Waist' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'waist_circumference'] = row['value_as_number']

    if 'hip' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'hip_circumference'] = row['value_as_number']

    if 'BMI' in row['standard_concept_name'] and row['value_as_number'] != 0:
        anthro_df.loc[anthro_df['person_id'] == row['person_id'], 'bmi'] = row['value_as_number']

## drop anthro_df index 0 (0 values)
anthro_df.drop(0, inplace=True)

# drop measurement_df duplicates. this df is not used again
measurement_df.drop_duplicates(subset=['person_id'], inplace=True)

## Begin Survey Encoding ##
# Define alcohol consumption mappings
# using Midpoint Method for annual frequency
alcohol_map = {
    'Drink Frequency Past Year: Monthly Or Less': 6,    # Midpoint of 0-1 per month * 12
    'Drink Frequency Past Year: 2 to 4 Per Month': 36,  # Midpoint of 2-4 per month * 12
    'Drink Frequency Past Year: 2 to 3 Per Week': 130,  # Midpoint of 2.5 per week * 52
    'Drink Frequency Past Year: 4 or More Per Week': 208, # Lower bound 4 per week * 52
    'PMI: Prefer Not To Answer': np.nan, # missing
    'PMI: Skip': np.nan,
    'PMI: Dont Know': np.nan
}

# Apply Mapping
# Initialize columns with NaN
survey_df['Drinks_per_Year'] = np.nan
survey_df['Pain_Med'] = 0
survey_df['Smoke_Years'] = 0

# Alcohol
# filter survey_df to only rows where question is
# about Alcohol first to avoid overwriting
alcohol_mask = survey_df['question'].str.contains('Alcohol', case=False, na=False)
survey_df.loc[alcohol_mask, 'Drinks_per_Year'] = survey_df.loc[alcohol_mask, 'answer'].map(alcohol_map)

# Pain Meds
meds_mask = survey_df['question'].str.contains('medications', case=False, na=False)
# check for 'Yes' or variants
survey_df.loc[meds_mask, 'Pain_Med'] = survey_df.loc[meds_mask, 'answer'].apply(
    lambda x: 1 if 'Yes' in str(x) or 'es' in str(x) else 0
)

# Smoking
smoke_mask = (~alcohol_mask) & (~meds_mask)

def clean_smoking_years(val):
    if str(val) in ['PMI: Prefer Not To Answer', 'PMI: Skip', 'PMI: Dont Know']:
        return np.nan
    try:
        return float(val)
    except:
        return 0

survey_df.loc[smoke_mask, 'Smoke_Years'] = survey_df.loc[smoke_mask, 'answer'].apply(clean_smoking_years)
## End Survey Encoding ##

# create empty medication df to collapse duplicate
# drug_df rows into unique rows
medication_data = {'person_id': '', 'ait_type': None, 'prior_tamoxifen': 0,
                   'pain_med': 0, 'prior_taxanes': 0, 'ait_start': datetime.now(tz=None)}
medication_df = pd.DataFrame(medication_data, index=[0])

## drop medication_df index 0 (0 values)
medication_df.drop(0, inplace=True)

# populate medication history
for index, row in drug_df.iterrows():
    if len(medication_df.loc[medication_df['person_id'] == row['person_id']]) == 0:
        new_row_df = pd.DataFrame([{'person_id': row['person_id'], 'prior_tamoxifen': 0,
                                    'pain_med': 0, 'prior_taxanes': 0, 'ait_start': datetime.now(tz=None)}])
        medication_df = pd.concat([medication_df, new_row_df], ignore_index=True)

for index, row in ait_df.iterrows():
    medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_start'] = pd.to_datetime(row['drug_exposure_start_datetime'])
    if 'anastrozole' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'anastrozole'
    if 'letrozole' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'letrozole'
    if 'exemestane' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'ait_type'] = 'exemestane'

for index, row in other_drugs_df.iterrows():
    if 'tamoxifen' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'prior_tamoxifen'] = 1
    elif 'paclitaxel' in row['drugs'] or 'docetaxel' in row['drugs'] or 'cabazitaxel' in row['drugs']:
        medication_df.loc[medication_df['person_id'] == row['person_id'], 'prior_taxanes'] = 1

print("Data cleanup complete.")

### END INITIAL DATA CLEANUP ###

In [ ]:
### BEGIN OUTLIER CHECKS ###

print("Checking outliers...")

## BMI ##
print("BMI > 50:")
print(len(anthro_df[anthro_df['bmi'] > 50]))

print("BMI < 10:")
print(len(anthro_df[(anthro_df['bmi'] > 0) & (anthro_df['bmi'] < 10)]))

print("BMI = 0:")
print(len(anthro_df[anthro_df['bmi'] == 0]))

## Height
print("Height > 250:")
print(len(anthro_df[anthro_df['height'] > 250]))

print("Height < 100:")
print(len(anthro_df[(anthro_df['height'] > 0) & (anthro_df['height'] < 100)]))

print("Height = 0:")
print(len(anthro_df[anthro_df['height'] == 0]))

## Waist circumference
print("Waist circumference > 200:")
print(len(anthro_df[anthro_df['waist_circumference'] > 200]))

print("Waist circumference < 50:")
print(len(anthro_df[(anthro_df['waist_circumference'] > 0) & (anthro_df['waist_circumference'] < 50)]))

print("Waist circumference = 0:")
print(len(anthro_df[anthro_df['waist_circumference'] == 0]))

## Hip circumference
print("Hip circumference > 250:")
print(len(anthro_df[anthro_df['hip_circumference'] > 250]))

print("Hip circumference < 50:")
print(len(anthro_df[(anthro_df['hip_circumference'] > 0) & (anthro_df['hip_circumference'] < 50)]))

print("Hip circumference = 0:")
print(len(anthro_df[anthro_df['hip_circumference'] == 0]))

print("Outlier check complete.")

### END OUTLIER CHECKS ###

In [ ]:
### BEGIN PRE-IMPUTATION DATA VISUALIZATION ###

print("Generating histograms for visualization...")

# Height distribution
plt.figure(figsize=(8, 6))
sns.histplot(anthro_df["height"], kde=True, bins=30, color='skyblue')
plt.title('Height Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Waist circumference distribution
plt.figure(figsize=(8, 6))
sns.histplot(anthro_df["waist_circumference"], kde=True, bins=30, color='skyblue')
plt.title('Waist Circumference Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

#Hip circumference distribution
plt.figure(figsize=(8, 6))
sns.histplot(anthro_df["hip_circumference"], kde=True, bins=30, color='skyblue')
plt.title('Hip Circumference Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# BMI distribution
plt.figure(figsize=(8, 6))
sns.histplot(anthro_df["bmi"], kde=True, bins=30, color='skyblue')
plt.title('BMI Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Drinks per Year distribution
plt.figure(figsize=(8, 6))
sns.histplot(survey_df["Drinks_per_Year"], kde=True, bins=30, color='skyblue')
plt.title('Drinks per Year Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Smoke Years distribution
plt.figure(figsize=(8, 6))
sns.histplot(survey_df["Smoke_Years"], kde=True, bins=30, color='skyblue')
plt.title('Smoke Years Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

print("Visualization complete.")

### END PRE-IMPUTATION DATA VISUALIZATION ###

In [ ]:
### BEGIN DATAFRAME MERGING ###

print("Merging dataframes and creating relevant sub-samples...")

# merge person and anthro dfs
person_anthro_df = pd.merge(person_df, anthro_df, how="left")

# merge with survey df
merge_1_df = pd.merge(person_anthro_df, survey_df, how="left")

# merge with condition df
merge_2_df = pd.merge(merge_1_df, condition_df, how='left')

# merge with medication df
joint_df = pd.merge(merge_2_df, medication_df, how='left')

# remove NAs from joint df
joint_df = joint_df[joint_df['ait_start'].notna()]

# populate pain, drugs, and age column in joint_df
# perform median imputation to 0 or missing values
for index, row in joint_df.iterrows():
    if pd.isna(row['pain']):
        joint_df.loc[index, 'pain'] = False
    else:
        if row['condition_start_datetime'].replace(tzinfo=None) < row['ait_start'].replace(tzinfo=None):
            joint_df.loc[index, 'pain'] = False
        else:
            joint_df.loc[index, 'pain'] = True

    if pd.isna(row['prior_tamoxifen']):
        joint_df.loc[index, 'prior_tamoxifen'] = 0

    if pd.isna(row['prior_taxanes']):
        joint_df.loc[index, 'prior_taxanes'] = 0

    if pd.isna(row['pain_med']):
        joint_df.loc[index, 'pain_med'] = 0

    joint_df.loc[index, 'age'] = float(calculate_age(row['date_of_birth'].replace(tzinfo=None), row['ait_start'].replace(tzinfo=None)))

print("Merging complete.")

### END DATAFRAME MERGING ###

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

### BEGIN IMPUTATION (MICE) ###
print("Starting multivariate imputations...")

cols_to_clean = ['height', 'waist_circumference', 'hip_circumference', 'bmi', "Drinks_per_Year",
                 "Smoke_Years", "prior_tamoxifen", "prior_taxanes"] # raw metrics

for col in cols_to_clean:
    # Set zeros to NaN
    joint_df[col] = joint_df[col].replace(0, np.nan)

    # Set extreme physiological outliers to NaN
    if col == 'bmi':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'waist_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'hip_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan
    if col == 'waist_circumference':
        joint_df.loc[(joint_df[col] == 0) | (joint_df[col] is None), col] = np.nan

# Select columns to use for imputation model
impute_cols = ['age', 'height', 'waist_circumference', 'hip_circumference', 'bmi', "Drinks_per_Year",
               "Smoke_Years", "prior_tamoxifen", "prior_taxanes"]

# Add race dummies to helper columns for better prediction
race_cols = [c for c in joint_df.columns if 'race_' in c]
prediction_matrix = joint_df[impute_cols + race_cols].copy()

# Initialize and run MICE (IterativeImputer)
mice_imputer = IterativeImputer(max_iter=10, random_state=42, sample_posterior=True)
imputed_data = mice_imputer.fit_transform(prediction_matrix)

# Put imputed values back into main df
imputed_df = pd.DataFrame(imputed_data, columns=prediction_matrix.columns, index=joint_df.index)

# --- QC step: Physiological limits
print("Clamping imputed values to physiological bounds...")

# Height
if 'height' in imputed_df.columns:
    imputed_df['height'] = imputed_df['height'].clip(lower=100, upper=250)

# Waist Circumference
if 'waist_circumference' in imputed_df.columns:
    imputed_df['waist_circumference'] = imputed_df['waist_circumference'].clip(lower=50, upper=200)

# Hip Circumference
if 'hip_circumference' in imputed_df.columns:
    imputed_df['hip_circumference'] = imputed_df['hip_circumference'].clip(lower=50, upper=250)

# BMI
if 'bmi' in imputed_df.columns:
    imputed_df['bmi'] = imputed_df['bmi'].clip(lower=10, upper=50)

# Drinks per Year
if 'Drinks_per_Year' in imputed_df.columns:
    imputed_df['Drinks_per_Year'] = imputed_df['Drinks_per_Year'].clip(lower=0, upper=208)

# Smoking Years
if 'Smoke_Years' in imputed_df.columns:
    imputed_df['Smoke_Years'] = imputed_df['Smoke_Years'].clip(lower=0, upper=67)

# -----------------------------------------------------------

# Update the main joint_df with the new clamped values
for col in impute_cols:
    joint_df[col] = imputed_df[col]

# Calculate indices based on imputed raw data
print("Re-calculating BRI and WHR based on imputed anthropometrics...")

for index, row in joint_df.iterrows():
    # Calculate BRI
    if row['waist_circumference'] > 0 and row['height'] > 0:
        joint_df.loc[index, 'bri'] = calculate_bri(row['waist_circumference'], row['height'])

    # Calculate WHR
    if row['waist_circumference'] > 0 and row['hip_circumference'] > 0:
        joint_df.loc[index, 'whr'] = row['waist_circumference'] / row['hip_circumference']

# Create sub-samples
# bri_df = joint_df.dropna(subset=['bri']).copy()
# whr_df = joint_df.dropna(subset=['whr']).copy()
bri_df = pd.DataFrame([p for p in joint_df.itertuples(index=True, name="PandasRow") if not pd.isna(p.bri)])
whr_df = pd.DataFrame([p for p in joint_df.itertuples(index=True, name="PandasRow") if not pd.isna(p.whr)])

print(f"Imputation Complete. Total Sample Size: {len(joint_df)}; BRI Sample Size: {len(bri_df)}; WHR Sample Size: {len(whr_df)}")

### END IMPUTATION (MICE)

In [ ]:
### BEGIN POST-IMPUTATION DATA VISUALIZATION ###

# Age distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["age"], kde=True, bins=30, color='skyblue')
plt.title('Age Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Height distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["height"], kde=True, bins=30, color='skyblue')
plt.title('Height Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Waist circumference distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["waist_circumference"], kde=True, bins=30, color='skyblue')
plt.title('Waist Circumference Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

#Hip circumference distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["hip_circumference"], kde=True, bins=30, color='skyblue')
plt.title('Hip Circumference Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# BMI distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["bmi"], kde=True, bins=30, color='skyblue')
plt.title('BMI Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# BRI distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["bri"], kde=True, bins=30, color='skyblue')
plt.title('BRI Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

#WHR distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["whr"], kde=True, bins=30, color='skyblue')
plt.title('WHR Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Drinks per Year distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["Drinks_per_Year"], kde=True, bins=30, color='skyblue')
plt.title('Drinks per Year Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

# Smoke Years distribution
plt.figure(figsize=(8, 6))
sns.histplot(joint_df["Smoke_Years"], kde=True, bins=30, color='skyblue')
plt.title('Smoke Years Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency / Density')
plt.show()

### END POST-IMPUTATION DATA VISUALIZATION ###

In [ ]:
from scipy.stats import chi2_contingency

print("\n" + "="*60)
print("  GLOBAL CHI-SQUARE TEST: RACE vs. JOINT PAIN")
print("="*60 + "\n")

# 1. Create contingency table
race_contingency_table = pd.crosstab(joint_df['race'], joint_df['pain'])

print("--- Contingency Table (Observed Counts) ---")
print(race_contingency_table)
print("\n")

# Run chi-square
chi2_stat, p_val, dof, expected = chi2_contingency(race_contingency_table)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom:   {dof}")
print(f"Global P-Value:       {p_val:.4e}")

# Interpretation
if p_val < 0.05:
    print("\nCONCLUSION: Statistically Significant Association (p < 0.05)")
    print("The distribution of joint pain is NOT the same across all racial groups.")
    print("Race is associated with the likelihood of having joint pain.")
else:
    print("\nCONCLUSION: No Statistically Significant Association (p >= 0.05)")
    print("The prevalence of joint pain does not significantly differ by race.")

In [ ]:
from scipy.stats import chi2_contingency

print("\n" + "="*60)
print("  GLOBAL CHI-SQUARE TEST: AROMATASE INHIBITOR TYPE vs. JOINT PAIN")
print("="*60 + "\n")

# Create contingency table
ait_contingency_table = pd.crosstab(joint_df['ait_type'], joint_df['pain'])

print("--- Contingency Table (Observed Counts) ---")
print(ait_contingency_table)
print("\n")

# Run chi-square
chi2_stat, p_val, dof, expected = chi2_contingency(ait_contingency_table)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom:   {dof}")
print(f"Global P-Value:       {p_val:.4e}")

# Interpretation
if p_val < 0.05:
    print("\nCONCLUSION: Statistically Significant Association (p < 0.05)")
    print("The distribution of joint pain is NOT the same across all AI types.")
    print("The specific Aromatase Inhibitor used is associated with the likelihood of having joint pain.")
else:
    print("\nCONCLUSION: No Statistically Significant Association (p >= 0.05)")
    print("The prevalence of joint pain does not significantly differ by Aromatase Inhibitor type.")

In [ ]:
### BEGIN DESCRIPTIVE STATISTICS ###

# create pain groups
pain = pd.DataFrame([p for p in joint_df.itertuples(index=True, name="PandasRow") if p.pain])
no_pain = pd.DataFrame([p for p in joint_df.itertuples(index=True, name="PandasRow") if not p.pain])

# create BRI pain groups
pain_bri = pd.DataFrame([p for p in bri_df.itertuples(index=True, name="PandasRow") if p.pain])
no_pain_bri = pd.DataFrame([p for p in bri_df.itertuples(index=True, name="PandasRow") if not p.pain])

# create WHR pain groups
pain_whr = pd.DataFrame([p for p in whr_df.itertuples(index=True, name="PandasRow") if p.pain])
no_pain_whr = pd.DataFrame([p for p in whr_df.itertuples(index=True, name="PandasRow") if not p.pain])

# summarize race data
pain_race_counts = pain['race'].value_counts()
no_pain_race_counts = no_pain['race'].value_counts()

pain_race_percentages = (pain['race'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)
no_pain_race_percentages = (no_pain['race'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)

pain_summary_df = pd.DataFrame({'Count': pain_race_counts, 'Percentage': pain_race_percentages})
no_pain_summary_df = pd.DataFrame({'Count': no_pain_race_counts, 'Percentage': no_pain_race_percentages})

# summarize ait type data
pain_ait_type_counts = pain['ait_type'].value_counts()
no_pain_ait_type_counts = no_pain['ait_type'].value_counts()

pain_ait_type_percentages = (pain['ait_type'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)
no_pain_ait_type_percentages = (no_pain['ait_type'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)

pain_ait_type_summary_df = pd.DataFrame({'Count': pain_ait_type_counts, 'Percentage': pain_ait_type_percentages})
no_pain_ait_type_summary_df = pd.DataFrame({'Count': no_pain_ait_type_counts, 'Percentage': no_pain_ait_type_percentages})

# Sample sizes
print("Cases:")
print(len(pain))

print("Controls:")
print(len(no_pain))

# Measures of Central Tendency - AGE
print("--- 2. Measures of Central Tendency ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Age ---")
    mean_val = df['age'].mean()
    median_val = df['age'].median()
    mode_val = df['age'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - AGE
print("--- 3. Measures of Variability ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Age ---")
    range_min = df['age'].min()
    range_max = df['age'].max()
    range_val = df['age'].max() - df['age'].min()
    variance_val = df['age'].var()
    std_dev_val = df['age'].std()
    print(f"Range:              {range_min} - {range_max}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Central Tendency - BMI
print("--- 4. Measures of Central Tendency ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: BMI ---")
    mean_val = df['bmi'].mean()
    median_val = df['bmi'].median()
    mode_val = df['bmi'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - BMI
print("--- 5. Measures of Variability - BMI ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: BMI ---")
    range_val = df['bmi'].max() - df['bmi'].min()
    variance_val = df['bmi'].var()
    std_dev_val = df['bmi'].std()
    print(f"Range:              {range_val:.2f}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Central Tendency - BRI
print("--- 6. Measures of Central Tendency ---")
for group, df in [('Pain', pain_bri), ('No Pain', no_pain_bri)]:
    print(f"\n--- {group}: BRI ---")
    mean_val = df['bri'].mean()
    median_val = df['bri'].median()
    mode_val = df['bri'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - BRI
print("--- 7. Measures of Variability - BRI ---")
for group, df in [('Pain', pain_bri), ('No Pain', no_pain_bri)]:
    print(f"\n--- {group}: BRI ---")
    range_val = df['bri'].max() - df['bri'].min()
    variance_val = df['bri'].var()
    std_dev_val = df['bri'].std()
    print(f"Range:              {range_val:.2f}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Central Tendency - WHR
print("--- 8. Measures of Central Tendency ---")
for group, df in [('Pain', pain_whr), ('No Pain', no_pain_whr)]:
    print(f"\n--- {group}: WHR ---")
    mean_val = df['whr'].mean()
    median_val = df['whr'].median()
    mode_val = df['whr'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - WHR
print("--- 9. Measures of Variability - WHR ---")
for group, df in [('Pain', pain_whr), ('No Pain', no_pain_whr)]:
    print(f"\n--- {group}: WHR ---")
    range_val = df['whr'].max() - df['whr'].min()
    variance_val = df['whr'].var()
    std_dev_val = df['whr'].std()
    print(f"Range:              {range_val:.2f}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Central Tendency - Smoke_Years
print("--- 10. Measures of Central Tendency ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Smoke_Years ---")
    mean_val = df['Smoke_Years'].mean()
    median_val = df['Smoke_Years'].median()
    mode_val = df['Smoke_Years'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - Smoke_Years
print("--- 11. Measures of Variability - Smoke_Years ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Smoke_Years ---")
    range_val = df['Smoke_Years'].max() - df['Smoke_Years'].min()
    variance_val = df['Smoke_Years'].var()
    std_dev_val = df['Smoke_Years'].std()
    print(f"Range:              {range_val:.2f}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Central Tendency - Drinks_per_Year
print("--- 12. Measures of Central Tendency ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Drinks_per_Year ---")
    mean_val = df['Drinks_per_Year'].mean()
    median_val = df['Drinks_per_Year'].median()
    mode_val = df['Drinks_per_Year'].mode()[0]
    print(f"Mean:   {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Mode:   {mode_val:.2f}")

# Measures of Variability - Drinks_per_Year
print("--- 13. Measures of Variability - Drinks_per_Year ---")
for group, df in [('Pain', pain), ('No Pain', no_pain)]:
    print(f"\n--- {group}: Drinks_per_Year ---")
    range_val = df['Drinks_per_Year'].max() - df['Drinks_per_Year'].min()
    variance_val = df['Drinks_per_Year'].var()
    std_dev_val = df['Drinks_per_Year'].std()
    print(f"Range:              {range_val:.2f}")
    print(f"Variance:           {variance_val:.2f}")
    print(f"Standard Deviation: {std_dev_val:.2f}")
print("\n" + "="*50 + "\n")

# Measures of Difference / Comparison
print("--- 14. Measures of Difference / Comparison ---")

# Independent Samples t-test (AGE)
t_stat, p_val_ttest = stats.ttest_ind(pain['age'], no_pain['age'])
print("\n--- Independent Samples t-test (Age) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Independent Samples t-test (BMI)
t_stat, p_val_ttest = stats.ttest_ind(pain['bmi'], no_pain['bmi'])
print("\n--- Independent Samples t-test (BMI) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Independent Samples t-test (BRI)
t_stat, p_val_ttest = stats.ttest_ind(pain_bri['bri'], no_pain_bri['bri'])
print("\n--- Independent Samples t-test (BRI) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Independent Samples t-test (WHR)
t_stat, p_val_ttest = stats.ttest_ind(pain_whr['whr'], no_pain_whr['whr'])
print("\n--- Independent Samples t-test (WHR) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Independent Samples t-test (Smoke_Years)
t_stat, p_val_ttest = stats.ttest_ind(pain['Smoke_Years'], no_pain['Smoke_Years'])
print("\n--- Independent Samples t-test (Smoke_Years) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Independent Samples t-test (Drinks_per_Year)
t_stat, p_val_ttest = stats.ttest_ind(pain['Drinks_per_Year'], no_pain['Drinks_per_Year'])
print("\n--- Independent Samples t-test (Drinks_per_Year) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("Result: The difference between the means is statistically significant.")
else:
    print("Result: The difference between the means is not statistically significant.")

# Race Data
print("\n--- Pain Race ---")
print("\n n=")
print(len(pain))
print("\n ")
print(pain_summary_df)
print("\n ")
print(pain_ait_type_summary_df)

print("\n--- No Pain Race ---")
print("\n n=")
print(len(no_pain))
print("\n ")
print(no_pain_summary_df)
print("\n ")
print(no_pain_ait_type_summary_df)

### END DESCRIPTIVE STATISTICS ###

In [ ]:
# 1. Log transform skewed variables
joint_df['log_drinks'] = np.log1p(joint_df['Drinks_per_Year'])
joint_df['log_smoke'] = np.log1p(joint_df['Smoke_Years'])
bri_df['log_drinks'] = np.log1p(bri_df['Drinks_per_Year'])
bri_df['log_smoke'] = np.log1p(bri_df['Smoke_Years'])
whr_df['log_drinks'] = np.log1p(whr_df['Drinks_per_Year'])
whr_df['log_smoke'] = np.log1p(whr_df['Smoke_Years'])

# 2. Standardization (z-scores)
scaler = StandardScaler()

# For joint df (BMI Analysis)
cols_to_scale_bmi = ['bmi', 'age', 'log_drinks', 'log_smoke']
joint_df[[c + '_z' for c in cols_to_scale_bmi]] = scaler.fit_transform(joint_df[cols_to_scale_bmi])

# For BRI df
cols_to_scale_bri = ['bri', 'age', 'log_drinks', 'log_smoke']
bri_df[[c + '_z' for c in cols_to_scale_bri]] = scaler.fit_transform(bri_df[cols_to_scale_bri])

# For WHR df
cols_to_scale_whr = ['whr', 'age', 'log_drinks', 'log_smoke']
whr_df[[c + '_z' for c in cols_to_scale_whr]] = scaler.fit_transform(whr_df[cols_to_scale_whr])

# 3. Categorical ref levels
reference_race = 'White'
reference_ait = 'anastrozole'

def get_dummies_with_ref(df, col_name, ref_level):
    dummies = pd.get_dummies(df[col_name], prefix=col_name, drop_first=False, dtype=int)
    ref_col = f"{col_name}_{ref_level}"
    if ref_col in dummies.columns:
        dummies.drop(columns=[ref_col], inplace=True) # Drop White to make it reference
    else:
        print(f"Warning: Reference level {ref_level} not found. Dropping first column.")
        return pd.get_dummies(df[col_name], prefix=col_name, drop_first=True, dtype=int)
    return dummies

race_dummies_joint = get_dummies_with_ref(joint_df, 'race', reference_race)
ait_dummies_joint = get_dummies_with_ref(joint_df, 'ait_type', reference_ait)
joint_df = pd.concat([joint_df, race_dummies_joint, ait_dummies_joint], axis=1)
joint_df = pd.DataFrame([p for p in joint_df.itertuples(index=True, name="PandasRow") if not pd.isna(p.bmi)])

race_dummies_bri = get_dummies_with_ref(bri_df, 'race', reference_race)
ait_dummies_bri = get_dummies_with_ref(bri_df, 'ait_type', reference_ait)
bri_df = pd.concat([bri_df, race_dummies_bri, ait_dummies_bri], axis=1)
bri_df = pd.DataFrame([p for p in bri_df.itertuples(index=True, name="PandasRow") if not pd.isna(p.bri)])

race_dummies_whr = get_dummies_with_ref(whr_df, 'race', reference_race)
ait_dummies_whr = get_dummies_with_ref(whr_df, 'ait_type', reference_ait)
whr_df = pd.concat([whr_df, race_dummies_whr, ait_dummies_whr], axis=1)
whr_df = pd.DataFrame([p for p in whr_df.itertuples(index=True, name="PandasRow") if not pd.isna(p.whr)])

### BEGIN LOGISTIC REGRESSIONS ###

# Helper function to clearly print ORs and P-values
def get_formatted_results(model_result, target_variable):
    """
    Extracts Odds Ratios, CIs, and P-values from a statsmodels result.
    Returns a clean DataFrame row for the target variable.
    """
    # Get coefs
    params = model_result.params

    # Get 95% CI
    conf = model_result.conf_int()
    conf.columns = ['2.5%', '97.5%']

    # Exponentiate to get ORs
    conf['Odds Ratio'] = np.exp(params)

    # Get p-values
    conf['P-value'] = model_result.pvalues

    # Reorder and return
    summary_df = conf[['Odds Ratio', '2.5%', '97.5%', 'P-value']]

    if target_variable in summary_df.index:
        return summary_df.loc[[target_variable]]
    else:
        return summary_df

print("\n" + "="*60)
print("  LOGISTIC REGRESSION RESULTS (CRUDE & ADJUSTED)")
print("="*60 + "\n")

outcome_var = 'pain'
race_cols = [c for c in joint_df.columns if 'race_' in c]
ait_cols = list(ait_dummies_joint.columns)

# MODEL: BMI
exposure_var = 'bmi_z'

# Crude BMI model
y = joint_df[outcome_var]
X = joint_df[exposure_var]
X = sm.add_constant(X)
result_bmi_crude = sm.Logit(y, X).fit(disp=0)

print(f"--- 1A. Crude BMI Model (Predictor: {exposure_var}) ---")
print(get_formatted_results(result_bmi_crude, exposure_var))
print("\n")

# Adjusted BMI model
confounders = ['age_z', 'log_smoke_z', 'log_drinks_z'] + race_cols + ait_cols
X_all = joint_df[[exposure_var] + confounders]
X_all = sm.add_constant(X_all)
result_bmi_all = sm.Logit(y, X_all).fit(disp=0)

print(f"--- 1B. Adjusted BMI Model (Predictor: {exposure_var}) ---")
print(get_formatted_results(result_bmi_all, exposure_var))
print("-" * 60 + "\n")


# MODEL: BRI
exposure_var_bri = 'bri_z'

# Crude BRI model
y_bri = bri_df[outcome_var]
X_bri = bri_df[exposure_var_bri]
X_bri = sm.add_constant(X_bri)
result_bri_crude = sm.Logit(y_bri, X_bri).fit(disp=0)

print(f"--- 2A. Crude BRI Model (Predictor: {exposure_var_bri}) ---")
print(get_formatted_results(result_bri_crude, exposure_var_bri))
print("\n")

# B. Adjusted BRI model
race_cols_bri = [c for c in bri_df.columns if 'race_' in c]
ait_cols_bri = list(ait_dummies_bri.columns)
confounders_bri = ['age_z', 'log_smoke_z', 'log_drinks_z'] + race_cols_bri + ait_cols_bri
X_all_bri = bri_df[[exposure_var_bri] + confounders_bri]
X_all_bri = sm.add_constant(X_all_bri)
result_bri_all = sm.Logit(y_bri, X_all_bri).fit(disp=0)

print(f"--- 2B. Adjusted BRI Model (Predictor: {exposure_var_bri}) ---")
print(get_formatted_results(result_bri_all, exposure_var_bri))
print("-" * 60 + "\n")


# MODEL: WHR
exposure_var_whr = 'whr_z'

# Crude WHR model
y_whr = whr_df[outcome_var]
X_whr = whr_df[exposure_var_whr]
X_whr = sm.add_constant(X_whr)
result_whr_crude = sm.Logit(y_whr, X_whr).fit(disp=0)

print(f"--- 3A. Crude WHR Model (Predictor: {exposure_var_whr}) ---")
print(get_formatted_results(result_whr_crude, exposure_var_whr))
print("\n")

# Adjusted WHR model
race_cols_whr = [c for c in whr_df.columns if 'race_' in c]
ait_cols_whr = list(ait_dummies_whr.columns)
confounders_whr = ['age_z', 'log_smoke_z', 'log_drinks_z'] + race_cols_whr + ait_cols_whr
X_all_whr = whr_df[[exposure_var_whr] + confounders_whr]
X_all_whr = sm.add_constant(X_all_whr)
result_whr_all = sm.Logit(y_whr, X_all_whr).fit(disp=0)

print(f"--- 3B. Adjusted WHR Model (Predictor: {exposure_var_whr}) ---")
print(get_formatted_results(result_whr_all, exposure_var_whr))
print("-" * 60 + "\n")


# --- ROC / AUC Analysis ---
from sklearn.metrics import roc_curve, roc_auc_score

def plot_roc(result_model, X_data, y_true, title):
    y_pred = result_model.predict(X_data)
    auc = roc_auc_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_pred)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], 'r--')
    plt.title(title)
    plt.legend()
    plt.show()
    print(f"{title} AUC: {auc:.4f}")

print("\n--- ROC / AUC ANALYSIS ---")
plot_roc(result_bmi_all, X_all, y, "BMI Adjusted ROC")
plot_roc(result_bri_all, X_all_bri, y_bri, "BRI Adjusted ROC")
plot_roc(result_whr_all, X_all_whr, y_whr, "WHR Adjusted ROC")